In [1]:
"""Stufe 1: Baselines und Validierungsaufbau (Bericht 3.3 bis 3.5).
Ausführen: uv run python src/baselines.py   (Ausgabe: reports/baselines_output.txt, reports/results_baselines.csv)
Vorher: src/clean.py muss gelaufen sein.
"""

'Stufe 1: Baselines und Validierungsaufbau (Bericht 3.3 bis 3.5).\nAusführen: uv run python src/baselines.py   (Ausgabe: reports/baselines_output.txt, reports/results_baselines.csv)\nVorher: src/clean.py muss gelaufen sein.\n'

In [2]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd().resolve().parent  # Skript oder Notebook
sys.path.insert(0, str(ROOT / "src"))
from forecast_eval import WINDOWS, load_clean, score, seasonal_baselines, targets_of, to_wide, window_frames  # noqa: E402

pd.set_option("display.width", 170)
lines = []


def out(msg=""):
    print(msg)
    lines.append(str(msg))


df = load_clean()
W_train = to_wide(df, "sales_train")  # maskiert: Spitzen, 25. Dezember, geschlossene Filialen = NaN
out(f"Reihen: {W_train.shape[1]}, Tage: {W_train.shape[0]}")

Reihen: 1782, Tage: 1688


## 1. Die drei Fenster
Pro Fenster ist der **Ursprung** der letzte bekannte Tag. Die Baselines sehen nur Daten bis dahin.
Danach folgen die 16 Tage, die wir vorhersagen und mit der Wahrheit vergleichen.

In [3]:
for name, origin in WINDOWS.items():
    t = targets_of(origin)
    out(f"{name}: Ursprung {origin.date()}, Prognose {t[0].date()} bis {t[-1].date()}")

W1 Aug 2017: Ursprung 2017-07-30, Prognose 2017-07-31 bis 2017-08-15
W2 Dez 2016: Ursprung 2016-12-08, Prognose 2016-12-09 bis 2016-12-24
W3 Mär 2017: Ursprung 2017-03-07, Prognose 2017-03-08 bis 2017-03-23


## 2. Baselines rechnen und bewerten
- **B1 Vorwoche:** gleicher Wochentag, jüngster bekannter Wert (mit Verfall bis 4 Wochen zurück)
- **B2 Mittel 4 Wochen:** Mittelwert der letzten 4 gleichen Wochentage

Bewertet wird nur, wo es einen echten Wert gibt, der Tag nicht ausgeschlossen ist (geschlossene Tage und Filialen)
und eine Prognose vorliegt. Zusätzlich zeigen wir den Fehler ohne die isolierten Spitzen.

In [4]:
rows, per_family = [], {}
for name, origin in WINDOWS.items():
    truth, excluded, outlier = window_frames(df, origin)
    preds = seasonal_baselines(W_train.loc[:origin], origin)
    for model, pred in preds.items():
        r = score(truth, pred, excluded, outlier)
        rows.append({"Fenster": name, "Modell": model, "RMSLE": r["rmsle"], "RMSLE ohne Spitzen": r["rmsle_ohne_spitzen"],
                     "bewertet": r["n_bewertet"], "ohne Prognose": r["n_ohne_prognose"], "ausgeschlossen": r["n_ausgeschlossen"]})
        per_family[(name, model)] = r["je_familie"]
res = pd.DataFrame(rows)
out("\n" + res.round(4).to_string(index=False))


    Fenster             Modell  RMSLE  RMSLE ohne Spitzen  bewertet  ohne Prognose  ausgeschlossen
W1 Aug 2017        B1 Vorwoche 0.6166              0.6166     28512              0               0
W1 Aug 2017 B2 Mittel 4 Wochen 0.5306              0.5306     28512              0               0
W2 Dez 2016        B1 Vorwoche 0.6469              0.6454     27885             99               0
W2 Dez 2016 B2 Mittel 4 Wochen 0.5938              0.5924     27885             99               0
W3 Mär 2017        B1 Vorwoche 0.5395              0.5393     27984              0               0
W3 Mär 2017 B2 Mittel 4 Wochen 0.4523              0.4520     27984              0               0


## 3. Benchmark
Als Vergleichswert für die späteren Modelle gilt je Fenster die **bessere** der beiden Baselines.
Das Erfolgskriterium (Bericht 3.4): mindestens 10 % niedrigerer RMSLE, im Mittel der drei Fenster
und in mindestens zwei von drei einzeln.

In [5]:
pivot = res.pivot(index="Fenster", columns="Modell", values="RMSLE").reindex(list(WINDOWS))
pivot["Benchmark (beste Baseline)"] = pivot.min(axis=1)
out("\nRMSLE je Fenster:\n" + pivot.round(4).to_string())
bench = pivot["Benchmark (beste Baseline)"]
out(f"\nMittel über die drei Fenster: {bench.mean():.4f}")
out(f"Zielwert für ein Modell (10 % besser): Mittel <= {0.9 * bench.mean():.4f}, "
    f"je Fenster <= {', '.join(f'{k.split()[0]}: {0.9 * v:.4f}' for k, v in bench.items())}")


RMSLE je Fenster:
Modell       B1 Vorwoche  B2 Mittel 4 Wochen  Benchmark (beste Baseline)
Fenster                                                                 
W1 Aug 2017       0.6166              0.5306                      0.5306
W2 Dez 2016       0.6469              0.5938                      0.5938
W3 Mär 2017       0.5395              0.4523                      0.4523

Mittel über die drei Fenster: 0.5256
Zielwert für ein Modell (10 % besser): Mittel <= 0.4730, je Fenster <= W1: 0.4775, W2: 0.5345, W3: 0.4071


## 4. Wo sind die Baselines schwach? Fehler je Produktfamilie (Fenster W1)
Ein Gesamtwert versteckt, dass einige Familien viel schlechter laufen. Das zeigt, wo ein Modell am meisten gewinnen kann.

In [6]:
fam = pd.DataFrame({m: per_family[("W1 Aug 2017", m)] for m in ["B1 Vorwoche", "B2 Mittel 4 Wochen"]})
fam["beste"] = fam.min(axis=1)
out("\nFehler je Familie, W1 (sortiert, schlechteste zuerst):\n" + fam.sort_values("beste", ascending=False).round(3).to_string())


Fehler je Familie, W1 (sortiert, schlechteste zuerst):
                            B1 Vorwoche  B2 Mittel 4 Wochen  beste
SCHOOL AND OFFICE SUPPLIES        1.812               1.799  1.799
LINGERIE                          0.945               0.710  0.710
GROCERY II                        0.878               0.683  0.683
HARDWARE                          0.738               0.603  0.603
CELEBRATION                       0.723               0.584  0.584
AUTOMOTIVE                        0.705               0.581  0.581
BEAUTY                            0.726               0.578  0.578
MAGAZINES                         0.714               0.576  0.576
SEAFOOD                           0.679               0.564  0.564
LADIESWEAR                        0.662               0.538  0.538
HOME AND KITCHEN I                0.636               0.525  0.525
PET SUPPLIES                      0.667               0.522  0.522
HOME AND KITCHEN II               0.609               0.512  0.512
PLAYER

## 5. Zwei Auffälligkeiten nachprüfen
a) In W2 gibt es 99 Zellen ohne Prognose: welche Reihen sind das?
b) `SCHOOL AND OFFICE SUPPLIES` hat im August einen Fehler von etwa 1,8, mehr als das Dreifache der anderen.
   Hypothese: Schulbeginn. Wir sehen uns den Monatsabsatz an.

In [7]:
origin = WINDOWS["W2 Dez 2016"]
truth, excluded, outlier = window_frames(df, origin)
pred = seasonal_baselines(W_train.loc[:origin], origin)["B2 Mittel 4 Wochen"]
missing = (truth.notna() & ~excluded & pred.isna()).sum()
missing = missing[missing > 0]
out("\nW2: Reihen mit Wahrheit, aber ohne Prognose (Filiale, Familie: Tage):")
out(missing.to_string())
first = df[df.sales_train.notna()].groupby(["store_nbr", "family"]).date.min()
out("Erster verwertbarer Wert dieser Reihen:\n" + first.reindex(missing.index).dt.date.to_string())


W2: Reihen mit Wahrheit, aber ohne Prognose (Filiale, Familie: Tage):
store_nbr  family                    
18         AUTOMOTIVE                    3
           BABY CARE                     3
           BEAUTY                        3
           BEVERAGES                     3
           BOOKS                         3
           BREAD/BAKERY                  3
           CELEBRATION                   3
           CLEANING                      3
           DAIRY                         3
           DELI                          3
           EGGS                          3
           FROZEN FOODS                  3
           GROCERY I                     3
           GROCERY II                    3
           HARDWARE                      3
           HOME AND KITCHEN I            3
           HOME AND KITCHEN II           3
           HOME APPLIANCES               3
           HOME CARE                     3
           LADIESWEAR                    3
           LAWN AND GARDEN     

In [8]:
school = df[df.family == "SCHOOL AND OFFICE SUPPLIES"].set_index("date").sales_train
monthly = school.groupby([school.index.year, school.index.month]).sum().unstack(0)
out("\nSCHOOL AND OFFICE SUPPLIES, Monatsabsatz (Zeilen = Monat, Spalten = Jahr):\n" + monthly.round(0).astype("Int64").to_string())


SCHOOL AND OFFICE SUPPLIES, Monatsabsatz (Zeilen = Monat, Spalten = Jahr):
date   2014   2015   2016   2017
date                            
1      1861      0   3108   2875
2         0      0   2815   1789
3      2235      0   3218   8246
4         0      0   8950  28689
5         0      0  10427   9658
6         0   1994   2461   2555
7      1958   1545   1746   8797
8         0  16148  17214  50169
9     18367  15719  23556   <NA>
10     3026   2927   3128   <NA>
11     2546   2612   2543   <NA>
12     1992   2392   2050   <NA>


In [9]:
res.to_csv(ROOT / "reports" / "results_baselines.csv", index=False)
(ROOT / "reports" / "baselines_output.txt").write_text("\n".join(lines), encoding="utf-8")

7647